# Streaming Stereo → 4.0 DSP Spatializer — Clean Workflow

这个 notebook 专门整理成一条清楚的工程管线：

```text
Input stereo L/R
→ Analyzer
→ Spatial Layer Extractor
→ Preset Resolver
   1. manual: 自己指定 preset
   2. auto_select: 规则自动选择一个现有 preset
   3. auto_acoustic: 根据当前曲子自动生成一个新 preset
→ Layer Router
→ 4.0 Renderer
→ Energy / Limiter
→ Export WAV + JSON
```

核心目标：你以后只需要改 **Cell 1** 里的 `PRESET_MODE` 和 `MANUAL_PRESET`，不用再到处找 cell。

## 1. 全局配置：输入路径、输出目录、preset 工作模式

In [ ]:
from pathlib import Path

# =========================
# Input / Output
# =========================
#INPUT_PATH = Path('/Users/saintpeter/Library/Mobile Documents/com~apple~CloudDocs/Downloads/Beat 2024 No.20 2.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/All The Things You Are.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Test Drive.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Hans Zimmer - Time (Official Audio).mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/lanjing.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/YE.mp3')
INPUT_PATH = Path('/Users/saintpeter/Downloads/little blue.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/一生所爱.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Love Blur.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Until I Found You.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Euphoria.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Drake.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/You Belong With Me.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Starboy.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/Redbone.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/孤勇者.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/跳楼机.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/一路向北.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/1.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/2.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/3.mp3')
#INPUT_PATH = Path('/Users/saintpeter/Downloads/radwimps.mp3')


OUT_DIR = Path("./spatializer_outputs_clean")
TARGET_SR = 48000
ANALYSIS_SECONDS = 2.0

OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Preset mode switch
# =========================
# 你现在有三种 preset 工作流：
#
# 1) "manual"
#    手动指定 MANUAL_PRESET。
#
# 2) "auto_select"
#    根据 analysis 规则，自动选择一个已有 preset。
#
# 3) "auto_acoustic"
#    不选择固定 preset，而是根据 analysis 动态生成 PRESETS["auto_acoustic"]。
#
# 推荐调试顺序：
#   manual → auto_select → auto_acoustic

#PRESET_MODE = "auto_acoustic" # "manual" / "auto_select" / "auto_acoustic"
PRESET_MODE = "manual"        # "manual" / "auto_select" / "auto_acoustic"
MANUAL_PRESET = "wide_smooth" # manual 模式下使用

# 如果你想批量测试，可以在最后的 batch cell 里跑多个 preset。
print("INPUT_PATH:", INPUT_PATH)
print("OUT_DIR:", OUT_DIR.resolve())
print("PRESET_MODE:", PRESET_MODE)
print("MANUAL_PRESET:", MANUAL_PRESET)

## 2. 依赖、基础工具、读写音频

In [ ]:
# 如缺库，取消注释运行：
# %pip install numpy scipy matplotlib soundfile librosa audioread

import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

EPS = 1e-9


def rms(x):
    x = np.asarray(x)
    return float(np.sqrt(np.mean(x * x) + EPS))


def peak(x):
    return float(np.max(np.abs(x)) + EPS)


def db(x):
    return 20.0 * np.log10(max(float(x), EPS))


def normalize_peak(x, peak_target=0.98):
    p = peak(x)
    if p > peak_target:
        return x * (peak_target / p)
    return x


def load_audio_any(path, target_sr=48000):
    """
    Load audio as stereo float32 at target_sr.

    Tries:
    1. soundfile
    2. librosa

    For MP3 on macOS, librosa may require ffmpeg/audioread support.
    """
    path = Path(path).expanduser()

    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    e_sf = None
    e_lr = None

    try:
        import soundfile as sf

        audio, sr = sf.read(str(path), always_2d=True, dtype="float32")

        if audio.shape[1] == 1:
            audio = np.repeat(audio, 2, axis=1)
        elif audio.shape[1] > 2:
            audio = audio[:, :2]

    except Exception as err_sf:
        e_sf = err_sf

        try:
            import librosa

            y, sr = librosa.load(str(path), sr=None, mono=False)

            if y.ndim == 1:
                audio = np.stack([y, y], axis=1).astype(np.float32)
            else:
                if y.shape[0] == 1:
                    audio = np.repeat(y, 2, axis=0).T.astype(np.float32)
                else:
                    audio = y[:2, :].T.astype(np.float32)

        except Exception as err_lr:
            e_lr = err_lr

            msg = f"""
Could not load audio.

Try installing dependencies inside the notebook:

    %pip install soundfile librosa audioread

If MP3 still fails on macOS, install ffmpeg:

    brew install ffmpeg

soundfile error:
    {e_sf}

librosa error:
    {e_lr}
"""
            raise RuntimeError(msg)

    if sr != target_sr:
        g = math.gcd(int(sr), int(target_sr))
        up = target_sr // g
        down = sr // g
        audio = signal.resample_poly(audio, up, down, axis=0).astype(np.float32)
        sr = target_sr

    audio = normalize_peak(audio, 0.99).astype(np.float32)
    return audio[:, 0], audio[:, 1], sr


def write_wav(path, audio, sr):
    import soundfile as sf

    path = Path(path).expanduser()
    path.parent.mkdir(parents=True, exist_ok=True)

    sf.write(str(path), audio.astype(np.float32), sr, subtype="FLOAT")
    return path


def butter_sos(kind, sr, cutoff, order=4):
    return signal.butter(order, cutoff, btype=kind, fs=sr, output="sos")


def filt_sos(x, sos):
    # Causal filtering: closer to streaming behavior than filtfilt.
    return signal.sosfilt(sos, x).astype(np.float32)


def lowpass(x, sr, cutoff, order=4):
    return filt_sos(x, butter_sos("lowpass", sr, cutoff, order))


def highpass(x, sr, cutoff, order=4):
    return filt_sos(x, butter_sos("highpass", sr, cutoff, order))


def bandpass(x, sr, low, high, order=4):
    return filt_sos(x, butter_sos("bandpass", sr, [low, high], order))


def band_split(x, sr):
    """
    5-band split:
    - bass:     <120 Hz
    - low_mid:  120–500 Hz
    - mid:      500–2000 Hz
    - high_mid: 2–6 kHz
    - air:      >6 kHz
    """
    return {
        "bass": lowpass(x, sr, 120),
        "low_mid": bandpass(x, sr, 120, 500),
        "mid": bandpass(x, sr, 500, 2000),
        "high_mid": bandpass(x, sr, 2000, 6000),
        "air": highpass(x, sr, 6000),
    }

## 3. 读取输入音频

In [ ]:
L, R, sr = load_audio_any(INPUT_PATH, TARGET_SR)
duration = len(L) / sr

print(f"Loaded: {INPUT_PATH}")
print(f"Sample rate: {sr}")
print(f"Duration: {duration:.2f} sec")
print(f"L RMS={rms(L):.5f}, R RMS={rms(R):.5f}, peak={max(peak(L), peak(R)):.5f}")

## 4. Streaming Analyzer：分析这首歌的声学结构

In [ ]:
def coherence(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    e_xy = np.mean(x * y)
    e_xx = np.mean(x * x)
    e_yy = np.mean(y * y)
    return float(abs(e_xy) / np.sqrt(e_xx * e_yy + EPS))


def transient_density(x, sr, hop=512):
    """
    Very lightweight transient estimate.
    It is not a beat detector. It just estimates how often strong short-term energy changes occur.
    """
    x = np.asarray(x)
    n = len(x)
    if n < hop * 4:
        return 0.0

    frames = []
    for i in range(0, n - hop, hop):
        frames.append(np.sqrt(np.mean(x[i:i + hop] ** 2) + EPS))
    frames = np.asarray(frames)

    diff = np.maximum(0.0, np.diff(frames))
    if len(diff) == 0:
        return 0.0

    threshold = np.mean(diff) + 1.5 * np.std(diff)
    return float(np.mean(diff > threshold))


def analyze_stereo(L, R, sr, analysis_seconds=2.0):
    n = min(len(L), int(sr * analysis_seconds))
    L0 = L[:n]
    R0 = R[:n]

    M = 0.70710678 * (L0 + R0)
    S = 0.70710678 * (L0 - R0)

    stereo_width = rms(S) / (rms(M) + rms(S) + EPS)
    center_dominance = rms(M) / (rms(M) + rms(S) + EPS)

    Lb = band_split(L0, sr)
    Rb = band_split(R0, sr)

    band_coh = {}
    band_side = {}
    for name in Lb.keys():
        Ml = 0.70710678 * (Lb[name] + Rb[name])
        Sl = 0.70710678 * (Lb[name] - Rb[name])
        band_coh[name] = coherence(Lb[name], Rb[name])
        band_side[name] = rms(Sl) / (rms(Ml) + rms(Sl) + EPS)

    # high_diffuse_ratio: useful rear ambience proxy.
    # Low coherence + high side ratio in high_mid/air means more natural rear material.
    high_diffuse_ratio = float(
        0.55 * (1.0 - band_coh["high_mid"]) * band_side["high_mid"]
        + 0.45 * (1.0 - band_coh["air"]) * band_side["air"]
    )

    mono_sum = 0.70710678 * (L0 + R0)
    trans = transient_density(mono_sum, sr)

    analysis = {
        "duration_analyzed_sec": n / sr,
        "stereo_width": float(stereo_width),
        "center_dominance": float(center_dominance),
        "bass_mono_ratio": float(band_coh["bass"]),
        "high_diffuse_ratio": float(high_diffuse_ratio),
        "transient_density": float(trans),
        "band_coherence": band_coh,
        "band_side_ratio": band_side,
    }

    return analysis


analysis = analyze_stereo(L, R, sr, ANALYSIS_SECONDS)
print(json.dumps(analysis, indent=2, ensure_ascii=False))

In [ ]:
# Analyzer visualization
fig, ax = plt.subplots(figsize=(9, 4))
labels = list(analysis["band_coherence"].keys())
coh_vals = [analysis["band_coherence"][k] for k in labels]
side_vals = [analysis["band_side_ratio"][k] for k in labels]

x = np.arange(len(labels))
w = 0.35

ax.bar(x - w / 2, coh_vals, width=w, label="coherence")
ax.bar(x + w / 2, side_vals, width=w, label="side ratio")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_title("Band-wise coherence and side ratio")
ax.legend()
plt.show()

## 5. Spatial Layer Extractor：拆出空间功能层

In [ ]:
def extract_spatial_layers(L, R, sr):
    """
    These are NOT clean stems.
    They are spatial-function layers:
    - bass: <120 Hz core
    - low_body: 120–500 Hz body/warmth
    - front_L / front_R: preserved front-facing stereo core
    - side_width: stereo difference bus
    - rear_ambience: side/diffuse ambience bus
    - high_air: high-frequency air bus
    """
    Lb = band_split(L, sr)
    Rb = band_split(R, sr)

    Mb = {name: 0.70710678 * (Lb[name] + Rb[name]) for name in Lb.keys()}
    Sb = {name: 0.70710678 * (Lb[name] - Rb[name]) for name in Lb.keys()}

    # True low bass: mostly front, only tiny quad-sharing later.
    bass = Mb["bass"]

    # Low-body: 120–500 Hz body/warmth. This is the safer way to make rear fuller.
    low_body = 0.95 * Mb["low_mid"] + 0.12 * Mb["mid"]

    # Front core: preserve original front brightness more faithfully.
    # Important: do not kill vocal air / acoustic brightness before 4ch rendering.
    front_L = Lb["low_mid"] + Lb["mid"] + 0.96 * Lb["high_mid"] + 0.62 * Lb["air"]
    front_R = Rb["low_mid"] + Rb["mid"] + 0.96 * Rb["high_mid"] + 0.62 * Rb["air"]

    # Side width: useful stereo spread.
    # Keep mid contribution moderate: too much 500–2000 Hz can spread vocal body.
    side_width = (
        0.05 * Sb["low_mid"]
        + 0.28 * Sb["mid"]
        + 0.92 * Sb["high_mid"]
        + 0.82 * Sb["air"]
    )

    # Rear ambience: mainly high-mid/air side, plus very tiny mid/air center fill.
    # This avoids rear being empty on narrow songs, while reducing vocal leakage.
    rear_ambience = (
        0.018 * Sb["mid"]
        + 0.88 * Sb["high_mid"]
        + 0.54 * Sb["air"]
        + 0.014 * Mb["high_mid"]
        + 0.018 * Mb["air"]
    )

    # High air: keep it lower than earlier rear_boost versions to avoid hiss/harshness.
    high_air = (
        0.76 * Sb["air"]
        + 0.08 * Mb["air"]
    )

    return {
        "bass": bass.astype(np.float32),
        "low_body": low_body.astype(np.float32),
        "front_L": front_L.astype(np.float32),
        "front_R": front_R.astype(np.float32),
        "side_width": side_width.astype(np.float32),
        "rear_ambience": rear_ambience.astype(np.float32),
        "high_air": high_air.astype(np.float32),
    }


layers = extract_spatial_layers(L, R, sr)

for name, x in layers.items():
    print(f"{name:16s} RMS={rms(x):.5f}, Peak={peak(x):.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
names = list(layers.keys())
vals = [rms(layers[n]) for n in names]

ax.bar(names, vals)
ax.set_title("Spatial layer RMS")
ax.tick_params(axis="x", rotation=45)
plt.show()

## 6. Preset Library：所有手动 preset 都集中放这里

In [ ]:
# ============================================================
# Preset Library
# 所有手动 preset 集中放这里。
# 注意：
# - manual 模式只会从这里拿 MANUAL_PRESET。
# - auto_select 模式会从这里选择一个已有 preset。
# - auto_acoustic 模式会自动生成 PRESETS["auto_acoustic"]。
# ============================================================

PRESETS = {
    "bypass": {
        "side_front": 0.0,
        "side_rear": 0.0,
        "amb_rear": 0.0,
        "air_rear": 0.0,
        "rear_master": 0.0,
        "decorrelation": 0.0,
        "rear_floor_ratio": 0.0,
        "max_rear_makeup": 1.0,
        "guard_scale": 1.0,
        "bass_quad": 0.0,
        "lowbody_rear": 0.0,
        "rear_air_gain": 1.0,
        "rear_highmid_gain": 1.0,
    },

    "ms_baseline": {
        "side_front": 0.0,
        "side_rear": 0.75,
        "amb_rear": 0.0,
        "air_rear": 0.0,
        "rear_master": 0.90,
        "decorrelation": 0.55,
        "rear_floor_ratio": 0.055,
        "max_rear_makeup": 2.5,
        "guard_scale": 1.0,
        "bass_quad": 0.0,
        "lowbody_rear": 0.0,
        "rear_air_gain": 0.50,
        "rear_highmid_gain": 0.85,
    },

    "general_pop_wide": {
        "side_front": 0.48,
        "side_rear": 0.92,
        "amb_rear": 0.78,
        "air_rear": 0.28,
        "rear_master": 0.96,
        "decorrelation": 0.38,
        "rear_floor_ratio": 0.095,
        "max_rear_makeup": 3.2,
        "guard_scale": 0.88,
        "bass_quad": 0.08,
        "lowbody_rear": 0.34,
        "rear_air_gain": 0.28,
        "rear_highmid_gain": 0.66,
    },

    "wide_smooth": {
        "side_front": 0.44,
        "side_rear": 1.05,
        "amb_rear": 1.18,
        "air_rear": 0.48,
        "rear_master": 1.06,
        "decorrelation": 0.60,
        "rear_floor_ratio": 0.125,
        "max_rear_makeup": 4.5,
        "guard_scale": 0.58,
        "bass_quad": 0.11,
        "lowbody_rear": 0.36,
        "rear_air_gain": 0.42,
        "rear_highmid_gain": 0.98,
    },

    "vocal_focus_wide": {
        "side_front": 0.50,
        "side_rear": 0.80,
        "amb_rear": 0.90,
        "air_rear": 0.18,
        "rear_master": 1.10,
        "decorrelation": 0.46,
        "rear_floor_ratio": 0.105,
        "max_rear_makeup": 3.6,
        "guard_scale": 0.82,
        "bass_quad": 0.08,
        "lowbody_rear": 0.42,
        "rear_air_gain": 0.36,
        "rear_highmid_gain": 0.80,
    },

    "vocal_room_body_clear": {
        "side_front": 0.52,
        "side_rear": 0.68,
        "amb_rear": 0.42,
        "air_rear": 0.12,
        "rear_master": 0.95,
        "decorrelation": 0.22,
        "rear_floor_ratio": 0.070,
        "max_rear_makeup": 2.0,
        "guard_scale": 1.35,
        "bass_quad": 0.05,
        "lowbody_rear": 0.44,
        "rear_air_gain": 0.12,
        "rear_highmid_gain": 0.24,
    },

    "bass_dry_wide": {
        "side_front": 0.56,
        "side_rear": 1.18,
        "amb_rear": 0.62,
        "air_rear": 0.32,
        "rear_master": 1.12,
        "decorrelation": 0.34,
        "rear_floor_ratio": 0.125,
        "max_rear_makeup": 4.0,
        "guard_scale": 0.90,
        "bass_quad": 0.13,
        "lowbody_rear": 0.52,
        "rear_air_gain": 0.34,
        "rear_highmid_gain": 0.76,
    },

    "epic_orchestral_depth": {
        "side_front": 0.48,
        "side_rear": 0.92,
        "amb_rear": 0.58,
        "air_rear": 0.26,
        "rear_master": 0.96,
        "decorrelation": 0.30,
        "rear_floor_ratio": 0.095,
        "max_rear_makeup": 3.0,
        "guard_scale": 0.76,
        "bass_quad": 0.07,
        "lowbody_rear": 0.34,
        "rear_air_gain": 0.26,
        "rear_highmid_gain": 0.58,
    },

    "vintage_jazz_room": {
        "side_front": 0.42,
        "side_rear": 1.12,
        "amb_rear": 0.72,
        "air_rear": 0.18,
        "rear_master": 1.08,
        "decorrelation": 0.44,
        "rear_floor_ratio": 0.145,
        "max_rear_makeup": 4.8,
        "guard_scale": 0.62,
        "bass_quad": 0.06,
        "lowbody_rear": 0.46,
        "rear_air_gain": 0.18,
        "rear_highmid_gain": 0.66,
    },
}

print("Available presets:")
for k in PRESETS.keys():
    print(" -", k)

## 7. Preset Resolver：三种 preset 工作流统一在这里处理

In [ ]:
def _clamp(x, lo, hi):
    return float(max(lo, min(hi, x)))


def _norm(x, lo, hi):
    return _clamp((float(x) - lo) / (hi - lo + EPS), 0.0, 1.0)


def _get(d, key, default=0.0):
    return float(d.get(key, default))


def auto_select_preset(analysis):
    """
    Mode 2: rule-based preset selection.
    只选择一个已有 preset，不动态生成参数。
    """
    width = analysis["stereo_width"]
    center = analysis["center_dominance"]
    bass_mono = analysis["bass_mono_ratio"]
    diffuse = analysis["high_diffuse_ratio"]
    transient = analysis["transient_density"]

    coh = analysis["band_coherence"]
    mid_coh = coh.get("mid", 0.0)
    highmid_coh = coh.get("high_mid", 0.0)

    # Extremely coherent vocal / mono-ish material:
    # clear vocal, little natural ambience.
    if center > 0.68 and mid_coh > 0.90 and highmid_coh > 0.88 and diffuse < 0.06:
        return "vocal_room_body_clear"

    # Bass-heavy dry music: rap / EDM / electronic
    if bass_mono > 0.85 and transient > 0.08 and diffuse < 0.12:
        return "bass_dry_wide"

    # Large diffuse hall / orchestral / cinematic
    if diffuse > 0.22 and width > 0.22 and transient < 0.20:
        return "epic_orchestral_depth"

    # Narrow vintage / small-field material
    if width < 0.22 and diffuse < 0.08 and mid_coh > 0.75:
        return "vintage_jazz_room"

    # Clear vocal but not fully mono-ish
    if center > 0.65 and mid_coh > 0.70:
        return "vocal_focus_wide"

    return "general_pop_wide"


def generate_auto_acoustic_preset(analysis, verbose=True):
    """
    Mode 3: dynamically generate a preset from acoustic features.
    不选择固定曲风 preset，而是实时填写一组参数。
    """
    width = _get(analysis, "stereo_width")
    center = _get(analysis, "center_dominance")
    bass_mono = _get(analysis, "bass_mono_ratio")
    diffuse = _get(analysis, "high_diffuse_ratio")
    transient = _get(analysis, "transient_density")

    coh = analysis.get("band_coherence", {})
    side = analysis.get("band_side_ratio", {})

    mid_coh = _get(coh, "mid")
    highmid_coh = _get(coh, "high_mid")

    lowmid_side = _get(side, "low_mid")
    mid_side = _get(side, "mid")
    highmid_side = _get(side, "high_mid")
    air_side = _get(side, "air")

    vocal_risk = (
        0.34 * _norm(center, 0.58, 0.82)
        + 0.26 * _norm(mid_coh, 0.72, 0.98)
        + 0.26 * _norm(highmid_coh, 0.70, 0.98)
        + 0.14 * (1.0 - _norm(diffuse, 0.03, 0.22))
    )
    vocal_risk = _clamp(vocal_risk, 0.0, 1.0)

    telephone_risk = (
        center > 0.66
        and mid_coh > 0.88
        and highmid_coh > 0.88
        and diffuse < 0.08
    )

    dry_bass_score = (
        0.40 * _norm(bass_mono, 0.78, 0.98)
        + 0.35 * _norm(transient, 0.06, 0.22)
        + 0.25 * (1.0 - _norm(diffuse, 0.04, 0.20))
    )
    dry_bass_score = _clamp(dry_bass_score, 0.0, 1.0)

    hall_score = (
        0.50 * _norm(diffuse, 0.16, 0.42)
        + 0.30 * _norm(width, 0.20, 0.45)
        + 0.20 * (1.0 - _norm(transient, 0.08, 0.28))
    )
    hall_score = _clamp(hall_score, 0.0, 1.0)

    narrow_score = (
        0.45 * (1.0 - _norm(width, 0.18, 0.36))
        + 0.35 * (1.0 - _norm(diffuse, 0.04, 0.18))
        + 0.20 * _norm((mid_coh + highmid_coh) * 0.5, 0.72, 0.98)
    )
    narrow_score = _clamp(narrow_score, 0.0, 1.0)

    side_material = _clamp(
        0.35 * _norm(width, 0.18, 0.42)
        + 0.25 * _norm(mid_side, 0.16, 0.38)
        + 0.25 * _norm(highmid_side, 0.18, 0.46)
        + 0.15 * _norm(air_side, 0.18, 0.48),
        0.0,
        1.0,
    )

    side_front = _clamp(0.44 + 0.14 * side_material - 0.04 * vocal_risk, 0.38, 0.62)

    side_rear = (
        0.70 + 0.36 * side_material + 0.18 * dry_bass_score
        + 0.14 * narrow_score - 0.22 * vocal_risk
    )
    side_rear = _clamp(side_rear, 0.48, 1.18)

    amb_rear = (
        0.46 + 0.28 * _norm(diffuse, 0.05, 0.28)
        + 0.12 * narrow_score - 0.18 * hall_score - 0.18 * vocal_risk
    )
    amb_rear = _clamp(amb_rear, 0.28, 0.88)

    air_rear = (
        0.16 + 0.16 * _norm(air_side, 0.18, 0.48)
        + 0.10 * _norm(diffuse, 0.08, 0.30) - 0.18 * vocal_risk
    )
    air_rear = _clamp(air_rear, 0.08, 0.38)

    rear_master = (
        0.88 + 0.16 * side_material + 0.08 * dry_bass_score
        + 0.08 * narrow_score - 0.06 * vocal_risk
    )
    rear_master = _clamp(rear_master, 0.78, 1.12)

    decorrelation = (
        0.28 + 0.12 * side_material + 0.08 * narrow_score
        - 0.12 * vocal_risk - 0.10 * hall_score
        - 0.08 * _norm(transient, 0.08, 0.25)
    )
    decorrelation = _clamp(decorrelation, 0.16, 0.46)

    rear_floor_ratio = (
        0.078 + 0.040 * narrow_score + 0.018 * dry_bass_score - 0.022 * vocal_risk
    )
    rear_floor_ratio = _clamp(rear_floor_ratio, 0.055, 0.145)

    max_rear_makeup = (
        2.4 + 1.6 * narrow_score + 0.8 * dry_bass_score - 0.8 * vocal_risk
    )
    max_rear_makeup = _clamp(max_rear_makeup, 1.6, 5.0)

    guard_scale = _clamp(0.72 + 0.62 * vocal_risk - 0.16 * side_material, 0.55, 1.45)

    bass_quad = (
        0.06 + 0.06 * dry_bass_score - 0.025 * vocal_risk - 0.020 * hall_score
    )
    bass_quad = _clamp(bass_quad, 0.035, 0.15)

    lowbody_rear = (
        0.28 + 0.18 * dry_bass_score + 0.16 * narrow_score
        + 0.08 * (1.0 - _norm(lowmid_side, 0.22, 0.42)) - 0.04 * vocal_risk
    )
    lowbody_rear = _clamp(lowbody_rear, 0.18, 0.58)

    rear_air_gain = (
        0.18 + 0.16 * _norm(diffuse, 0.08, 0.32)
        + 0.08 * _norm(air_side, 0.18, 0.48) - 0.20 * vocal_risk
    )
    rear_air_gain = _clamp(rear_air_gain, 0.08, 0.40)

    rear_highmid_gain = (
        0.46 + 0.18 * side_material + 0.08 * _norm(highmid_side, 0.18, 0.46)
        - 0.34 * vocal_risk - 0.10 * hall_score
    )
    rear_highmid_gain = _clamp(rear_highmid_gain, 0.18, 0.78)

    # Safety overrides
    if telephone_risk:
        # Do not over-expand raw vocal presence to rear,
        # but keep enough perceptual brightness so the rear pair is audible.
        side_rear = min(side_rear, 0.92)
        amb_rear = min(amb_rear, 0.52)

        # 原来 0.16 / 0.08 太暗，会吞高频和空间轮廓
        air_rear = min(max(air_rear, 0.14), 0.24)

        # 不能太高，否则前后 delay / comb 会回来
        decorrelation = min(max(decorrelation, 0.24), 0.32)

        # 后方保底稍微提高，但不要靠音量硬撑
        rear_floor_ratio = min(max(rear_floor_ratio, 0.095), 0.125)
        max_rear_makeup = min(max(max_rear_makeup, 3.0), 4.2)

        # 后方 air 不能低到 0.08，否则后方没有可感知轮廓
        rear_air_gain = min(max(rear_air_gain, 0.20), 0.34)

        # 2–6k 仍然保护，但不要压到 0.18 那么死
        rear_highmid_gain = min(max(rear_highmid_gain, 0.34), 0.48)

        # 人声仍然保护中心
        guard_scale = max(guard_scale, 1.10)

        # 保持 120–500Hz 空间体积
        lowbody_rear = max(lowbody_rear, 0.46)

        
    if hall_score > 0.65:
        amb_rear = min(amb_rear, 0.58)
        decorrelation = min(decorrelation, 0.30)
        rear_highmid_gain = min(rear_highmid_gain, 0.58)
        rear_air_gain = min(rear_air_gain, 0.30)

    if dry_bass_score > 0.65 and not telephone_risk:
        lowbody_rear = max(lowbody_rear, 0.48)
        bass_quad = max(bass_quad, 0.10)
        amb_rear = min(amb_rear, 0.62)
        decorrelation = min(decorrelation, 0.34)

    auto_preset = {
        "side_front": side_front,
        "side_rear": side_rear,
        "amb_rear": amb_rear,
        "air_rear": air_rear,
        "rear_master": rear_master,
        "decorrelation": decorrelation,
        "rear_floor_ratio": rear_floor_ratio,
        "max_rear_makeup": max_rear_makeup,
        "guard_scale": guard_scale,
        "bass_quad": bass_quad,
        "lowbody_rear": lowbody_rear,
        "rear_air_gain": rear_air_gain,
        "rear_highmid_gain": rear_highmid_gain,
    }

    auto_info = {
        "vocal_risk": vocal_risk,
        "telephone_risk": bool(telephone_risk),
        "dry_bass_score": dry_bass_score,
        "hall_score": hall_score,
        "narrow_score": narrow_score,
        "side_material": side_material,
        "rear_presence_fill": 0.16 if narrow_score > 0.75 and side_material < 0.20 else 0.06,
        "rear_brightness_fill": 0.10 if narrow_score > 0.75 and side_material < 0.20 else 0.04,
    }

    if verbose:
        print("=== Auto Acoustic Preset Generated ===")
        for k, v in auto_info.items():
            print(f"{k:18s}: {v}")
        print("\nPreset values:")
        for k, v in auto_preset.items():
            print(f"{k:18s}: {v:.4f}")

    return auto_preset, auto_info


def resolve_preset(PRESET_MODE, MANUAL_PRESET, analysis, PRESETS):
    """
    三种 preset 工作流统一入口。
    Returns:
        preset_name: 最终使用的 preset 名字
        preset_mode_used: manual / auto_select / auto_acoustic
        auto_info: auto_acoustic 的分析分数；其他模式为空 dict
    """
    mode = PRESET_MODE.strip().lower()
    auto_info = {}

    if mode == "manual":
        preset_name = MANUAL_PRESET
        if preset_name not in PRESETS:
            raise KeyError(f"MANUAL_PRESET={preset_name!r} not found in PRESETS.")
        return preset_name, "manual", auto_info

    if mode == "auto_select":
        preset_name = auto_select_preset(analysis)
        if preset_name not in PRESETS:
            raise KeyError(f"auto_select selected {preset_name!r}, but it is not in PRESETS.")
        return preset_name, "auto_select", auto_info

    if mode == "auto_acoustic":
        auto_preset, auto_info = generate_auto_acoustic_preset(analysis, verbose=True)
        PRESETS["auto_acoustic"] = auto_preset
        return "auto_acoustic", "auto_acoustic", auto_info

    raise ValueError("PRESET_MODE must be 'manual', 'auto_select', or 'auto_acoustic'.")


preset_name, preset_mode_used, auto_info = resolve_preset(PRESET_MODE, MANUAL_PRESET, analysis, PRESETS)

print("\nResolved preset:")
print("  mode:", preset_mode_used)
print("  preset:", preset_name)

## 8. Layer Router：把 preset 转成最终 routing

In [ ]:
def route_layers(analysis, preset_name, apply_analysis_adaptation=True):
    """
    Convert preset values into final routing values.

    For manual / auto_select:
        apply lightweight analysis adaptation.

    For auto_acoustic:
        values are already generated from analysis, so we skip most adaptation.
    """
    if preset_name not in PRESETS:
        raise KeyError(f"Preset not found: {preset_name}")

    p = dict(PRESETS[preset_name])

    if preset_name in ["bypass", "ms_baseline"]:
        p["preset_name"] = preset_name
        return p

    if preset_name == "auto_acoustic":
        apply_analysis_adaptation = False

    center = analysis["center_dominance"]
    high_diffuse = analysis["high_diffuse_ratio"]
    transient = analysis["transient_density"]
    width = analysis["stereo_width"]

    guard_scale = float(p.get("guard_scale", 1.0))

    if apply_analysis_adaptation:
        # Center guard:
        # strong center vocal/main material should reduce dangerous rear high-mid / air routing.
        center_guard = np.clip((center - 0.52) / 0.35, 0.0, 1.0)

        p["side_rear"] *= (1.0 - 0.10 * guard_scale * center_guard)
        p["amb_rear"] *= (1.0 - 0.08 * guard_scale * center_guard)
        p["air_rear"] *= (1.0 - 0.14 * guard_scale * center_guard)

        # Width boost:
        # if source has usable stereo width, let side contribute to rear.
        width_boost = np.clip(0.85 + width / 0.40, 0.85, 1.45)
        p["side_rear"] *= width_boost

        # Diffuse boost:
        # high diffuse material can feed rear, but do not over-boost air.
        diffuse_boost = np.clip(1.0 + 1.4 * high_diffuse, 1.0, 1.35)
        p["amb_rear"] *= diffuse_boost
        p["air_rear"] *= np.clip(1.0 + 0.7 * high_diffuse, 1.0, 1.20)

        # Transient guard:
        # reduce decorrelation and ambience for punch preservation.
        transient_guard = np.clip(transient / 0.05, 0.0, 1.0)
        p["decorrelation"] *= (1.0 - 0.14 * guard_scale * transient_guard)
        p["amb_rear"] *= (1.0 - 0.05 * guard_scale * transient_guard)
        p["air_rear"] *= (1.0 - 0.08 * guard_scale * transient_guard)

        # Low-body safety
        p["lowbody_rear"] *= (1.0 - 0.12 * guard_scale * center_guard)
        p["lowbody_rear"] *= (1.0 - 0.10 * guard_scale * transient_guard)
        p["bass_quad"] *= (1.0 - 0.18 * guard_scale * transient_guard)

    else:
        center_guard = np.clip((center - 0.52) / 0.35, 0.0, 1.0)
        transient_guard = np.clip(transient / 0.05, 0.0, 1.0)
        width_boost = 1.0
        diffuse_boost = 1.0

    # Clip all final values for safety
    for k in ["side_front", "side_rear", "amb_rear", "air_rear", "rear_master", "decorrelation", "bass_quad", "lowbody_rear"]:
        p[k] = float(np.clip(p.get(k, 0.0), 0.0, 1.8))

    p["rear_floor_ratio"] = float(np.clip(p.get("rear_floor_ratio", 0.0), 0.0, 0.30))
    p["max_rear_makeup"] = float(np.clip(p.get("max_rear_makeup", 1.0), 1.0, 8.0))
    p["rear_air_gain"] = float(np.clip(p.get("rear_air_gain", 1.0), 0.08, 1.0))
    p["rear_highmid_gain"] = float(np.clip(p.get("rear_highmid_gain", 1.0), 0.18, 1.10))
    p["bass_quad"] = float(np.clip(p.get("bass_quad", 0.0), 0.0, 0.25))
    p["lowbody_rear"] = float(np.clip(p.get("lowbody_rear", 0.0), 0.0, 0.60))

    p["guard_scale"] = guard_scale
    p["preset_name"] = preset_name
    p["preset_mode_used"] = preset_mode_used
    p["center_guard"] = float(center_guard)
    p["transient_guard"] = float(transient_guard)
    p["width_boost"] = float(width_boost)
    p["diffuse_boost"] = float(diffuse_boost)

    return p


routing = route_layers(
    analysis,
    preset_name,
    apply_analysis_adaptation=(preset_mode_used != "auto_acoustic")
)

print(json.dumps(routing, indent=2, ensure_ascii=False))

## 9. Decorrelator 与 rear tone shaping

In [ ]:
def delay_samples(x, d):
    d = int(max(0, d))
    if d == 0:
        return x.copy()
    return np.concatenate([np.zeros(d, dtype=x.dtype), x[:-d]]).astype(np.float32)


def allpass_delay(x, delay, g):
    delay = int(max(1, delay))
    b = np.zeros(delay + 1, dtype=np.float64)
    a = np.zeros(delay + 1, dtype=np.float64)
    b[0] = -g
    b[delay] = 1.0
    a[0] = 1.0
    a[delay] = -g
    return signal.lfilter(b, a, x).astype(np.float32)


def soften_rear_tone(x, sr, air_gain=0.50, highmid_gain=0.95):
    """
    Rear-only tone shaping.
    Keeps the front timbre intact while controlling rear harshness.
    """
    b = band_split(x, sr)
    y = (
        b["bass"]
        + b["low_mid"]
        + b["mid"]
        + highmid_gain * b["high_mid"]
        + air_gain * b["air"]
    )
    return y.astype(np.float32)


def decorrelate_rear(x, sr, amount=0.6):
    amount = float(np.clip(amount, 0.0, 1.0))

    # Shorter delay/all-pass to reduce audible echo / combing.
    d_l = int(sr * (0.0055 + 0.0030 * amount))
    d_r = int(sr * (0.0085 + 0.0040 * amount))

    xl = delay_samples(x, d_l)
    xr = delay_samples(-x, d_r)

    g1 = 0.28 + 0.24 * amount
    g2 = 0.20 + 0.20 * amount

    yl = allpass_delay(xl, int(sr * 0.0037), g1)
    yl = allpass_delay(yl, int(sr * 0.0089), g2)

    yr = allpass_delay(xr, int(sr * 0.0049), g1 * 0.92)
    yr = allpass_delay(yr, int(sr * 0.0113), g2 * 0.88)

    return yl.astype(np.float32), yr.astype(np.float32)

## 10. 4.0 Renderer：把 layers 渲染成 LF/RF/LB/RB

In [ ]:
def apply_rear_floor(out4, routing, preset_name):
    """
    Avoid almost-silent rear channels, but do not over-amplify hiss.
    """
    if preset_name == "bypass":
        return out4

    front_rms = rms(out4[:, :2])
    rear_rms = rms(out4[:, 2:])

    rear_floor_ratio = float(routing.get("rear_floor_ratio", 0.0))
    max_makeup = float(routing.get("max_rear_makeup", 1.0))

    target_rear_rms = front_rms * rear_floor_ratio

    if rear_floor_ratio > 0 and rear_rms < target_rear_rms:
        rear_gain = target_rear_rms / (rear_rms + EPS)
        rear_gain = float(np.clip(rear_gain, 1.0, max_makeup))
        out4 = out4.copy()
        out4[:, 2:] *= rear_gain
        print(
            f"Applied rear makeup gain: {rear_gain:.2f}x "
            f"(target={rear_floor_ratio:.3f}, front_rms={front_rms:.5f}, rear_before={rear_rms:.5f})"
        )

    return out4.astype(np.float32)


def render_4ch(L, R, layers, routing, sr, preset_name):
    if preset_name == "bypass":
        return np.stack([L, R, np.zeros_like(L), np.zeros_like(R)], axis=1).astype(np.float32)

    if preset_name == "ms_baseline":
        S = 0.70710678 * (L - R)
        LB, RB = decorrelate_rear(S, sr, routing["decorrelation"])
        LF = L.copy()
        RF = R.copy()

        LB *= routing["rear_master"] * routing["side_rear"]
        RB *= routing["rear_master"] * routing["side_rear"]

        LB = soften_rear_tone(LB, sr, routing.get("rear_air_gain", 0.65), routing.get("rear_highmid_gain", 0.95))
        RB = soften_rear_tone(RB, sr, routing.get("rear_air_gain", 0.65), routing.get("rear_highmid_gain", 0.95))

        out = np.stack([LF, RF, LB, RB], axis=1).astype(np.float32)
        return apply_rear_floor(out, routing, preset_name)

    bass = layers["bass"]
    low_body = layers.get("low_body", np.zeros_like(bass))
    front_L = layers["front_L"]
    front_R = layers["front_R"]
    side = layers["side_width"]
    amb = layers["rear_ambience"]
    air = layers["high_air"]

    bass_quad = float(routing.get("bass_quad", 0.0))
    lowbody_rear = float(routing.get("lowbody_rear", 0.0))

    # Bass distribution:
    # front-only equal-power bass: LF/RF each 0.7071
    # quad equal-power bass: LF/RF/LB/RB each 0.5
    bass_front_gain = (1.0 - bass_quad) * 0.7071 + bass_quad * 0.5
    bass_rear_gain = bass_quad * 0.5

    LF = bass_front_gain * bass + front_L + routing["side_front"] * side
    RF = bass_front_gain * bass + front_R - routing["side_front"] * side

    rear_low_body = lowbody_rear * low_body

    rear_base = (
        routing["side_rear"] * side
        + routing["amb_rear"] * amb
        + routing["air_rear"] * air
    )

    LB, RB = decorrelate_rear(rear_base, sr, routing["decorrelation"])
    LB *= routing["rear_master"]
    RB *= routing["rear_master"]

    # Add controlled bass/low-body support after decorrelated rear generation.
    LB += bass_rear_gain * bass + rear_low_body
    RB += bass_rear_gain * bass + rear_low_body

    # Rear-only tone shaping.
    LB = soften_rear_tone(LB, sr, routing.get("rear_air_gain", 0.50), routing.get("rear_highmid_gain", 0.95))
    RB = soften_rear_tone(RB, sr, routing.get("rear_air_gain", 0.50), routing.get("rear_highmid_gain", 0.95))

    out = np.stack([LF, RF, LB, RB], axis=1).astype(np.float32)
    out = apply_rear_floor(out, routing, preset_name)
    return out.astype(np.float32)


raw_4ch = render_4ch(L, R, layers, routing, sr, preset_name)

print("Raw 4ch shape:", raw_4ch.shape)
print("Raw rear/front:", rms(raw_4ch[:, 2:]) / (rms(raw_4ch[:, :2]) + EPS), f"({db(rms(raw_4ch[:, 2:]) / (rms(raw_4ch[:, :2]) + EPS)):.2f} dB)")

## 11. Energy Manager + Limiter

In [ ]:
def match_energy_to_input(stereo_L, stereo_R, out4, max_boost_db=1.0, max_cut_db=-3.0):
    input_energy = np.mean(stereo_L**2 + stereo_R**2) + EPS
    output_energy = np.mean(np.sum(out4**2, axis=1)) + EPS

    target_gain = np.sqrt(input_energy / output_energy)

    min_gain = 10 ** (max_cut_db / 20.0)
    max_gain = 10 ** (max_boost_db / 20.0)

    gain = float(np.clip(target_gain, min_gain, max_gain))
    return (out4 * gain).astype(np.float32), gain


def safety_limiter(out4, ceiling=0.98):
    p = peak(out4)
    if p <= ceiling:
        return out4.astype(np.float32), 1.0

    gain = ceiling / p
    return (out4 * gain).astype(np.float32), float(gain)


energy_matched_4ch, energy_gain = match_energy_to_input(L, R, raw_4ch)
final_4ch, limiter_gain = safety_limiter(energy_matched_4ch, ceiling=0.98)

print(f"Energy gain: {energy_gain:.3f} ({db(energy_gain):.2f} dB)")
print(f"Limiter gain: {limiter_gain:.3f} ({db(limiter_gain):.2f} dB)")
print(f"Final peak: {peak(final_4ch):.4f}")

## 12. 导出 4ch WAV、stereo preview、diagnostics JSON

In [ ]:
stem = INPUT_PATH.stem.replace(" ", "_")
out_4ch_path = OUT_DIR / f"{stem}_{preset_name}_4ch.wav"
preview_path = OUT_DIR / f"{stem}_{preset_name}_preview_stereo.wav"
diag_path = OUT_DIR / f"{stem}_{preset_name}_diagnostics.json"

write_wav(out_4ch_path, final_4ch, sr)

# Stereo preview only for quick headphone checking.
# It is not the true 4.0 experience.
preview_L = final_4ch[:, 0] + 0.50 * final_4ch[:, 2]
preview_R = final_4ch[:, 1] + 0.50 * final_4ch[:, 3]
preview = np.stack([preview_L, preview_R], axis=1).astype(np.float32)
preview = normalize_peak(preview, 0.98)
write_wav(preview_path, preview, sr)

front_energy = float(np.mean(final_4ch[:, 0]**2 + final_4ch[:, 1]**2))
rear_energy = float(np.mean(final_4ch[:, 2]**2 + final_4ch[:, 3]**2))
total_energy = front_energy + rear_energy + EPS

channel_rms = {
    "LF": rms(final_4ch[:, 0]),
    "RF": rms(final_4ch[:, 1]),
    "LB": rms(final_4ch[:, 2]),
    "RB": rms(final_4ch[:, 3]),
}

channel_peak = {
    "LF": peak(final_4ch[:, 0]),
    "RF": peak(final_4ch[:, 1]),
    "LB": peak(final_4ch[:, 2]),
    "RB": peak(final_4ch[:, 3]),
}

rear_to_front_rms_ratio = rms(final_4ch[:, 2:]) / (rms(final_4ch[:, :2]) + EPS)

diagnostics = {
    "input_file": str(INPUT_PATH),
    "preset_mode_requested": PRESET_MODE,
    "manual_preset_requested": MANUAL_PRESET,
    "preset_mode_used": preset_mode_used,
    "preset_used": preset_name,
    "sample_rate": sr,
    "duration_seconds": duration,
    "analysis": analysis,
    "auto_info": auto_info,
    "routing": routing,
    "energy_gain": energy_gain,
    "limiter_gain": limiter_gain,
    "output": {
        "front_energy_ratio": front_energy / total_energy,
        "rear_energy_ratio": rear_energy / total_energy,
        "rear_to_front_rms_ratio": rear_to_front_rms_ratio,
        "rear_to_front_db": db(rear_to_front_rms_ratio),
        "channel_rms": channel_rms,
        "channel_peak": channel_peak,
        "peak": peak(final_4ch),
        "rms": rms(final_4ch),
        "out_4ch_path": str(out_4ch_path),
        "preview_path": str(preview_path),
    },
}

with open(diag_path, "w", encoding="utf-8") as f:
    json.dump(diagnostics, f, indent=2, ensure_ascii=False)

print("Wrote 4ch:", out_4ch_path)
print("Wrote preview:", preview_path)
print("Wrote diagnostics:", diag_path)
print("Preset mode:", preset_mode_used)
print("Preset used:", preset_name)
print("Front energy ratio:", diagnostics["output"]["front_energy_ratio"])
print("Rear energy ratio:", diagnostics["output"]["rear_energy_ratio"])
print("Rear/Front RMS ratio:", f"{rear_to_front_rms_ratio:.4f}", f"({db(rear_to_front_rms_ratio):.2f} dB)")
print("Channel RMS:", channel_rms)

## 13. 快速试听：普通 stereo preview，不代表真实 4.0

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

print("Original stereo:")
display(Audio(np.stack([L, R], axis=0), rate=sr))

print("Preview stereo downmix:")
preview_audio, preview_sr = sf.read(str(preview_path), always_2d=True)
display(Audio(preview_audio.T, rate=preview_sr))

## 14. 诊断图：通道 RMS 与前后能量比例

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
names = ["LF", "RF", "LB", "RB"]
vals = [channel_rms[n] for n in names]

ax.bar(names, vals)
ax.set_title(f"Channel RMS — {preset_name}")
ax.set_ylabel("RMS")
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["front", "rear"], [front_energy / total_energy, rear_energy / total_energy])
ax.set_ylim(0, 1)
ax.set_title("Front / Rear energy ratio")
plt.show()

print("Rear/Front dB:", db(rear_to_front_rms_ratio))

## 15. 批量导出多个 preset 做 A/B 测试

In [ ]:
# 可控导出当前 preset / 批量 preset 的 4.0 与 binaural A/B：
# 默认只导出“当前 preset 的纯 4ch”，避免一次运行生成大量文件。
#
# 输出文件：
# - *_4ch.wav: 原始 LF/RF/LB/RB 四声道
# - *_binaural_4p0.wav: direct/procedural 4.0 虚拟扬声器 binaural
# - *_binaural_front_pair.wav: direct/procedural，只保留 LF/RF 的 binaural
# - *_binaural_rear_pair.wav: direct/procedural，只保留 LB/RB 的 binaural
# - *_binaural_4p0_room_rir.wav: direct binaural 再经过小型偏干房间 stereo RIR/matrix convolution
# - *_binaural_front_pair_room_rir.wav: front pair + small/dry room RIR
# - *_binaural_rear_pair_room_rir.wav: rear pair + small/dry room RIR
#
# 说明：
# 1) direct/procedural HRTF preview 不等同于个性化 HRTF/SOFA/BRIR。
# 2) 这里的 room RIR 是 deterministic synthetic small/dry room RIR，不联网下载外部 IR；
#    它被加在 binaural 输出之后，用于测试“房间外化/反射”是否改善后方扬声器感。
# 3) 这仍不是严格 BRIR：真正 BRIR 应该是每个 speaker -> 双耳的一对实测/模拟 IR。
# 4) 后方/front pair 单独导出时会独立 peak normalize，方便听 localization，而不是听音量差。

BINAURAL_FRONT_AZIMUTH_DEG = 30.0
BINAURAL_REAR_AZIMUTH_DEG = 135.0
BINAURAL_FULL_REAR_GAIN_DB = 1.5

# =========================
# Export switches
# =========================
# 默认安全模式：只跑当前 preset，只导出纯 4ch。
# 需要耳机监听时，把 EXPORT_BINAURAL_DIRECT 或 EXPORT_BINAURAL_ROOM_RIR 改成 True。
BATCH_ALL_PRESETS = False
EXPORT_4CH = True
EXPORT_BINAURAL_DIRECT = False
EXPORT_BINAURAL_ROOM_RIR = True
EXPORT_ROOM_RIR_MATRIX = False  # True 时额外导出 LL/LR/RL/RR 四列 RIR wav 供检查
BINAURAL_VARIANTS = ("4p0", "front_pair", "rear_pair")  # 可改成 ("4p0",) 来进一步减少文件

# 小一点、干一点的房间 RIR 参数：近似 small control room / dry studio booth。
# RIR 以 2x2 stereo matrix 形式应用到已经生成的 binaural：
# out_L = in_L * LL + in_R * RL
# out_R = in_L * LR + in_R * RR
ROOM_RIR_ENABLED = True
ROOM_RIR_NAME = "synthetic_small_dry_room_stereo_matrix"
ROOM_RIR_RT60_SECONDS = 0.32
ROOM_RIR_LENGTH_SECONDS = 0.45
ROOM_RIR_LATE_START_SECONDS = 0.040
ROOM_RIR_RANDOM_SEED = 20260611
ROOM_RIR_KEEP_TAIL = True

BINAURAL_LAYOUT = {
    "LF": {"channel": 0, "azimuth": -BINAURAL_FRONT_AZIMUTH_DEG, "is_rear": False},
    "RF": {"channel": 1, "azimuth": BINAURAL_FRONT_AZIMUTH_DEG, "is_rear": False},
    "LB": {"channel": 2, "azimuth": -BINAURAL_REAR_AZIMUTH_DEG, "is_rear": True},
    "RB": {"channel": 3, "azimuth": BINAURAL_REAR_AZIMUTH_DEG, "is_rear": True},
}


def peak_normalize_exact(audio, peak_target=0.98):
    p = peak(audio)
    if p <= EPS:
        return audio.astype(np.float32)
    return (audio * (peak_target / p)).astype(np.float32)


def rear_pinna_cue(x, sr):
    """
    Lightweight rear-position cue for headphones.
    Generic HRTF has strong front/back confusion, so the rear pair gets a
    mild 6-9 kHz pinna notch plus a tiny early body reflection.
    """
    notch = bandpass(x, sr, 5600, 9200, order=2)
    air = highpass(x, sr, 9000, order=2)
    presence = bandpass(x, sr, 1800, 4200, order=2)
    early_body = delay_samples(lowpass(x, sr, 2600, order=2), int(0.012 * sr))

    y = x - 0.30 * notch - 0.06 * air + 0.035 * presence + 0.055 * early_body
    return y.astype(np.float32)


def far_ear_shadow(x, sr, side_amount):
    """Frequency-dependent far-ear attenuation for a simple HRTF preview."""
    low = lowpass(x, sr, 1500, order=2)
    high = (x - low).astype(np.float32)

    low_gain = 10 ** ((-0.7 * side_amount) / 20.0)
    high_gain = 10 ** ((-7.5 * side_amount) / 20.0)
    return (low_gain * low + high_gain * high).astype(np.float32)


def render_virtual_speaker_binaural(x, sr, azimuth_deg, is_rear=False):
    """
    Render one virtual speaker into headphone L/R.
    azimuth convention: negative = left, positive = right.
    """
    x = np.asarray(x, dtype=np.float32)
    if is_rear:
        x = rear_pinna_cue(x, sr)

    side = float(np.sin(np.deg2rad(azimuth_deg)))
    side_amount = abs(side)
    itd_samples = int(round(sr * 0.00068 * side_amount))

    near = x
    far = delay_samples(far_ear_shadow(x, sr, side_amount), itd_samples)

    if side < 0:
        left, right = near, far
    elif side > 0:
        left, right = far, near
    else:
        left, right = near, near

    return np.stack([left, right], axis=1).astype(np.float32)


def render_4ch_binaural(out4, sr, active_channels=("LF", "RF", "LB", "RB"), rear_gain_db=0.0):
    out4 = np.asarray(out4, dtype=np.float32)
    y = np.zeros((out4.shape[0], 2), dtype=np.float32)
    rear_gain = 10 ** (rear_gain_db / 20.0)

    for name in active_channels:
        spec = BINAURAL_LAYOUT[name]
        mono = out4[:, spec["channel"]]
        if spec["is_rear"]:
            mono = mono * rear_gain

        y += render_virtual_speaker_binaural(
            mono,
            sr,
            spec["azimuth"],
            is_rear=spec["is_rear"],
        )

    return y.astype(np.float32)


def _safe_add_tap(h, sample_index, in_ch, out_ch, gain):
    if 0 <= sample_index < h.shape[0]:
        h[sample_index, in_ch, out_ch] += float(gain)


def make_small_dry_room_stereo_rir(
    sr,
    rt60=ROOM_RIR_RT60_SECONDS,
    length_seconds=ROOM_RIR_LENGTH_SECONDS,
    late_start_seconds=ROOM_RIR_LATE_START_SECONDS,
    seed=ROOM_RIR_RANDOM_SEED,
):
    """
    Deterministic synthetic small/dry-room stereo matrix RIR.

    Shape: (samples, input_channel, output_ear)
      h[:, 0, 0] = input L -> output L (LL)
      h[:, 0, 1] = input L -> output R (LR)
      h[:, 1, 0] = input R -> output L (RL)
      h[:, 1, 1] = input R -> output R (RR)

    Compared with the previous medium-room version, this one is smaller/drier:
      - shorter RT60 and shorter exported tail
      - earlier but lower-level reflections
      - quieter late tail so it does not wash out HRTF localization cues
    """
    sr = int(sr)
    n = max(8, int(round(length_seconds * sr)))
    h = np.zeros((n, 2, 2), dtype=np.float32)

    # Direct binaural path: do not collapse/crossfeed direct localization.
    h[0, 0, 0] = 1.0
    h[0, 1, 1] = 1.0

    # Small/dry room early reflections. Lower level than the previous medium-room RIR.
    # Tuple: (delay_seconds, gain_db, lateral_bias)
    early_taps = [
        (0.0042, -22.0, -0.60),  # close side wall, dry
        (0.0067, -23.5, 0.52),
        (0.0108, -25.0, 0.08),   # floor/ceiling
        (0.0165, -27.0, -0.28),
        (0.0240, -29.0, 0.34),
        (0.0330, -31.0, -0.12),
    ]

    for delay_s, gain_db, lateral in early_taps:
        d = int(round(delay_s * sr))
        g = 10 ** (gain_db / 20.0)

        # Same-side reflections.
        _safe_add_tap(h, d, 0, 0, g * (1.00 - 0.05 * lateral))
        _safe_add_tap(h, d + int(round(0.00025 * sr)), 1, 1, g * (1.00 + 0.05 * lateral))

        # Cross-ear reflections are later/quieter to avoid excessive wetness and image blur.
        cross_delay = d + int(round((0.0019 + 0.0015 * abs(lateral)) * sr))
        cross_gain = g * (0.22 + 0.10 * abs(lateral))
        _safe_add_tap(h, cross_delay, 0, 1, cross_gain * (1.00 + 0.04 * lateral))
        _safe_add_tap(h, cross_delay + int(round(0.00035 * sr)), 1, 0, cross_gain * (1.00 - 0.04 * lateral))

    # Quiet, dark, decorrelated late tail.
    late_start = min(n - 1, max(1, int(round(late_start_seconds * sr))))
    tail_len = n - late_start
    if tail_len > 0:
        rng = np.random.default_rng(seed)
        t = np.arange(tail_len, dtype=np.float32) / float(sr)
        decay = (10.0 ** (-3.0 * t / max(float(rt60), 0.05))).astype(np.float32)

        for in_ch in range(2):
            for out_ch in range(2):
                noise = rng.standard_normal(tail_len).astype(np.float32)
                # Drier/darker tail: less top-end sheen, no rumble/DC.
                noise = lowpass(noise, sr, 5200, order=2)
                noise = highpass(noise, sr, 150, order=2)
                noise = noise / (rms(noise) + EPS)

                same_side = in_ch == out_ch
                late_gain = 0.012 if same_side else 0.0065
                h[late_start:, in_ch, out_ch] += (late_gain * decay * noise).astype(np.float32)

    return h.astype(np.float32)

def room_rir_matrix_to_4ch(room_ir):
    """Export helper: columns are LL, LR, RL, RR."""
    return np.stack(
        [
            room_ir[:, 0, 0],
            room_ir[:, 0, 1],
            room_ir[:, 1, 0],
            room_ir[:, 1, 1],
        ],
        axis=1,
    ).astype(np.float32)


def convolve_1d_full(x, h):
    x = np.asarray(x, dtype=np.float32)
    h = np.asarray(h, dtype=np.float32)
    if hasattr(signal, "oaconvolve"):
        y = signal.oaconvolve(x, h, mode="full")
    else:
        y = signal.fftconvolve(x, h, mode="full")
    return y.astype(np.float32)


def apply_room_rir_to_binaural(binaural, room_ir, keep_tail=ROOM_RIR_KEEP_TAIL):
    """
    Apply stereo matrix RIR to a 2-channel binaural signal.

    This is intentionally post-binaural room convolution. It can improve externalization,
    but it is not equivalent to per-speaker BRIR convolution.
    """
    binaural = np.asarray(binaural, dtype=np.float32)
    if binaural.ndim != 2 or binaural.shape[1] != 2:
        raise ValueError(f"Expected binaural shape (n, 2), got {binaural.shape}")

    y_l = (
        convolve_1d_full(binaural[:, 0], room_ir[:, 0, 0])
        + convolve_1d_full(binaural[:, 1], room_ir[:, 1, 0])
    )
    y_r = (
        convolve_1d_full(binaural[:, 0], room_ir[:, 0, 1])
        + convolve_1d_full(binaural[:, 1], room_ir[:, 1, 1])
    )

    out_len = len(y_l) if keep_tail else binaural.shape[0]
    return np.stack([y_l[:out_len], y_r[:out_len]], axis=1).astype(np.float32)


def write_binaural_ab_files(
    final4,
    sr,
    stem,
    preset_name,
    room_ir=None,
    write_direct=True,
    write_room_rir=False,
    variants=BINAURAL_VARIANTS,
):
    variants = tuple(variants)
    valid_variants = {"4p0", "front_pair", "rear_pair"}
    invalid = [v for v in variants if v not in valid_variants]
    if invalid:
        raise ValueError(f"Unknown binaural variant(s): {invalid}. Valid: {sorted(valid_variants)}")
    if (write_direct or write_room_rir) and not variants:
        raise ValueError("BINAURAL_VARIANTS is empty while binaural export is enabled")
    if write_room_rir and room_ir is None:
        raise ValueError("write_room_rir=True requires a room_ir")

    direct_path_templates = {
        "4p0": OUT_DIR / f"{stem}_{preset_name}_binaural_4p0.wav",
        "front_pair": OUT_DIR / f"{stem}_{preset_name}_binaural_front_pair.wav",
        "rear_pair": OUT_DIR / f"{stem}_{preset_name}_binaural_rear_pair.wav",
    }
    room_path_templates = {
        "4p0": OUT_DIR / f"{stem}_{preset_name}_binaural_4p0_room_rir.wav",
        "front_pair": OUT_DIR / f"{stem}_{preset_name}_binaural_front_pair_room_rir.wav",
        "rear_pair": OUT_DIR / f"{stem}_{preset_name}_binaural_rear_pair_room_rir.wav",
    }
    render_specs = {
        "4p0": {
            "active_channels": ("LF", "RF", "LB", "RB"),
            "rear_gain_db": BINAURAL_FULL_REAR_GAIN_DB,
            "metric_prefix": "binaural_4p0",
        },
        "front_pair": {
            "active_channels": ("LF", "RF"),
            "rear_gain_db": 0.0,
            "metric_prefix": "front_pair",
        },
        "rear_pair": {
            "active_channels": ("LB", "RB"),
            "rear_gain_db": 0.0,
            "metric_prefix": "rear_pair",
        },
    }

    paths = {}
    metrics = {}

    for key in variants:
        spec = render_specs[key]
        direct = render_4ch_binaural(
            final4,
            sr,
            active_channels=spec["active_channels"],
            rear_gain_db=spec["rear_gain_db"],
        )
        if key == "4p0":
            direct = normalize_peak(direct, 0.98).astype(np.float32)
        else:
            direct = peak_normalize_exact(direct, 0.98)

        prefix = spec["metric_prefix"]
        metrics[f"{prefix}_peak"] = peak(direct)
        metrics[f"{prefix}_rms"] = rms(direct)

        if write_direct:
            paths[key] = direct_path_templates[key]
            write_wav(paths[key], direct, sr)

        if write_room_rir:
            room = apply_room_rir_to_binaural(direct, room_ir)
            if key == "4p0":
                room = normalize_peak(room, 0.98).astype(np.float32)
            else:
                room = peak_normalize_exact(room, 0.98)

            room_key = f"{key}_room_rir"
            paths[room_key] = room_path_templates[key]
            write_wav(paths[room_key], room, sr)
            metrics[f"{prefix}_room_rir_peak"] = peak(room)
            metrics[f"{prefix}_room_rir_rms"] = rms(room)

    if write_room_rir:
        metrics["room_rir_added_tail_seconds"] = float(ROOM_RIR_LENGTH_SECONDS if ROOM_RIR_KEEP_TAIL else 0.0)

    return paths, metrics

BATCH_TEST_PRESETS = [
    "bypass",
    "ms_baseline",
    "general_pop_wide",
    "wide_smooth",
    "vocal_focus_wide",
    "vocal_room_body_clear",
    "bass_dry_wide",
    "epic_orchestral_depth",
    "vintage_jazz_room",
]

if BATCH_ALL_PRESETS:
    # 如果当前 preset 是 auto_acoustic 或自定义 preset，也放到第一项一起导出。
    TEST_PRESETS = list(dict.fromkeys([preset_name] + BATCH_TEST_PRESETS))
else:
    TEST_PRESETS = [preset_name]

if not (EXPORT_4CH or EXPORT_BINAURAL_DIRECT or EXPORT_BINAURAL_ROOM_RIR):
    raise ValueError("At least one export flag must be True: EXPORT_4CH / EXPORT_BINAURAL_DIRECT / EXPORT_BINAURAL_ROOM_RIR")

room_ir = None
room_rir_path = None
room_rir_config = None
if EXPORT_BINAURAL_ROOM_RIR:
    if not ROOM_RIR_ENABLED:
        raise ValueError("EXPORT_BINAURAL_ROOM_RIR=True requires ROOM_RIR_ENABLED=True")
    room_ir = make_small_dry_room_stereo_rir(sr)
    if EXPORT_ROOM_RIR_MATRIX:
        room_rir_path = OUT_DIR / f"{stem}_{ROOM_RIR_NAME}_rir_matrix_LL_LR_RL_RR.wav"
        write_wav(room_rir_path, room_rir_matrix_to_4ch(room_ir), sr)
    room_rir_config = {
        "name": ROOM_RIR_NAME,
        "path": str(room_rir_path) if room_rir_path is not None else None,
        "matrix_channel_order": "LL, LR, RL, RR",
        "rt60_seconds": float(ROOM_RIR_RT60_SECONDS),
        "length_seconds": float(ROOM_RIR_LENGTH_SECONDS),
        "late_start_seconds": float(ROOM_RIR_LATE_START_SECONDS),
        "keep_tail": bool(ROOM_RIR_KEEP_TAIL),
        "seed": int(ROOM_RIR_RANDOM_SEED),
        "note": "Synthetic post-binaural small/dry stereo matrix RIR; useful for externalization A/B, not a measured per-speaker BRIR.",
    }

print("Selected presets:", TEST_PRESETS)
print("Export 4ch:", EXPORT_4CH)
print("Export binaural direct:", EXPORT_BINAURAL_DIRECT, "variants:", BINAURAL_VARIANTS)
print("Export binaural room RIR:", EXPORT_BINAURAL_ROOM_RIR)
print("Binaural layout azimuths:", {k: v["azimuth"] for k, v in BINAURAL_LAYOUT.items()})
print("Full 4.0 binaural rear monitor gain:", f"{BINAURAL_FULL_REAR_GAIN_DB:.1f} dB")
if room_rir_config is not None:
    print(
        "Room RIR:",
        ROOM_RIR_NAME,
        f"RT60={ROOM_RIR_RT60_SECONDS:.2f}s",
        f"length={ROOM_RIR_LENGTH_SECONDS:.2f}s",
        "matrix:",
        room_rir_path if room_rir_path is not None else "not exported",
    )

batch_manifest = []

for p_name in TEST_PRESETS:
    route = route_layers(analysis, p_name, apply_analysis_adaptation=(p_name != "auto_acoustic"))
    raw = render_4ch(L, R, layers, route, sr, p_name)
    matched, eg = match_energy_to_input(L, R, raw)
    final, lg = safety_limiter(matched)

    out_path = None
    if EXPORT_4CH:
        out_path = OUT_DIR / f"{stem}_{p_name}_4ch.wav"
        write_wav(out_path, final, sr)

    binaural_paths = {}
    binaural_metrics = {}
    if EXPORT_BINAURAL_DIRECT or EXPORT_BINAURAL_ROOM_RIR:
        binaural_paths, binaural_metrics = write_binaural_ab_files(
            final,
            sr,
            stem,
            p_name,
            room_ir=room_ir,
            write_direct=EXPORT_BINAURAL_DIRECT,
            write_room_rir=EXPORT_BINAURAL_ROOM_RIR,
            variants=BINAURAL_VARIANTS,
        )

    front_r = rms(final[:, :2])
    rear_r = rms(final[:, 2:])
    rear_front_ratio = rear_r / (front_r + EPS)

    batch_manifest.append({
        "preset": p_name,
        "four_channel_path": str(out_path) if out_path is not None else None,
        "binaural_paths": {k: str(v) for k, v in binaural_paths.items()},
        "room_rir": room_rir_config,
        "rear_to_front_rms_ratio": float(rear_front_ratio),
        "rear_to_front_db": float(db(rear_front_ratio)),
        "energy_gain": float(eg),
        "limiter_gain": float(lg),
        "binaural_metrics": binaural_metrics,
    })

    print(f"\n{p_name}")
    if out_path is not None:
        print("  4ch:", out_path)
    else:
        print("  4ch: skipped")

    if binaural_paths:
        for key, path in binaural_paths.items():
            print(f"  binaural {key}:", path)
    else:
        print("  binaural: skipped")

    print(
        "  peak=", f"{peak(final):.3f}",
        "rear/front=", f"{rear_front_ratio:.4f}",
        f"({db(rear_front_ratio):.2f} dB)",
        "energy_gain=", f"{eg:.3f}",
        "limiter_gain=", f"{lg:.3f}",
    )

manifest_path = OUT_DIR / f"{stem}_batch_export_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(batch_manifest, f, indent=2, ensure_ascii=False)

print("\nBatch export manifest:", manifest_path)


## 16. 工作流说明

现在 preset 有三种方式：

### A. 手动选择 preset

在 Cell 1 设置：

```python
PRESET_MODE = "manual"
MANUAL_PRESET = "wide_smooth"
```

适合你做主观 A/B 调音。

### B. 规则自动选择已有 preset

```python
PRESET_MODE = "auto_select"
```

它会根据 `analysis` 选择：
- `vocal_room_body_clear`
- `bass_dry_wide`
- `epic_orchestral_depth`
- `vintage_jazz_room`
- `vocal_focus_wide`
- `general_pop_wide`

### C. 自动生成声学 preset

```python
PRESET_MODE = "auto_acoustic"
```

它不套用某一个固定 preset，而是根据当前曲子的：
- center vocal risk
- telephone/plastic risk
- dry bass score
- hall score
- narrow score
- useful side material

动态生成 `PRESETS["auto_acoustic"]`。